<a href="https://colab.research.google.com/github/AkankshaB123/python/blob/main/Price_Simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [88]:
import pandas as pd
import statsmodels.api as sm

# -----------------------------
# 1. Load data
# -----------------------------
# Replace with your actual file
# Load the raw data into 'raw_df' to preserve the original DataFrame state
raw_df = pd.read_csv("/content/sample_data/Case_Study_Urgency_Message_Data.xlsx - Aggregated (1).csv", low_memory=False)
# For compatibility, df still points to a copy of raw_df, but subsequent operations should ideally use raw_df directly or its copies
df = raw_df.copy()

In [89]:
df.head(2)

,#,ADR_USD,Unnamed: 2,hotel_id,city_id,City,star_rating,accommadation_type_name,chain_hotel,booking_date,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75
0,"17,800",4.26,0.13%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/2/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"21,585",4.62,0.15%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/23/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
df.columns

Index(['#', 'ADR_USD', 'Unnamed: 2', 'hotel_id', 'city_id', 'City',
       'star_rating', 'accommadation_type_name', 'chain_hotel', 'booking_date',
       'checkin_date', 'checkout_date', 'Stay_duration', 'Week_num',
       'Month_no', 'Date_Diff', 'Date_Range', 'Month_Year', 'Week_name',
       'booking_type', 'Rating_bucket', 'Price bucket', 'Week_day_no',
       'Unnamed: 23', 'Assumption: Every line item is a unique booking',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35',
       'Core Objective: Should Agoda implement urgency messaging based on how prices behave as check-in approaches? If yes, how and where? --> Yes, but selectively',
       'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40',
       'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44',
       'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',
       'Unnam

In [96]:
import pandas as pd
import statsmodels.api as sm

# -----------------------------
# 2. Select only needed columns
# -----------------------------
# Create df_for_modeling from raw_df to ensure it always uses the original data
df_for_modeling = raw_df[['ADR_USD', 'star_rating', 'Date_Diff', 'city_id', 'City', 'accommadation_type_name', 'Price bucket']].copy()

# Drop missing values if any
df_for_modeling = df_for_modeling.dropna()

In [92]:
df.head(2)

,#,ADR_USD,Unnamed: 2,hotel_id,city_id,City,star_rating,accommadation_type_name,chain_hotel,booking_date,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,Unnamed: 71,Unnamed: 72,Unnamed: 73,Unnamed: 74,Unnamed: 75
0,"17,800",4.26,0.13%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/2/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"21,585",4.62,0.15%,"582,528","9,395",City_A,2.0,Hotel,non-chain,12/23/2016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [97]:
# Start with a fresh copy of the DataFrame with selected columns
df = df_for_modeling.copy()

# Ordinal mapping for 'Price bucket'
price_bucket_mapping = {'Luxury': 2, 'Budget friendly': 1, 'Low': 0}
df['Price_bucket_encoded'] = df['Price bucket'].map(price_bucket_mapping)

# Drop rows where Price_bucket_encoded resulted in NaN due to unmapped categories
df = df.dropna(subset=['Price_bucket_encoded'])

# Perform one-hot encoding on the 'City' and 'accommadation_type_name' columns with drop_first=False
df = pd.get_dummies(df, columns=['City', 'accommadation_type_name'], drop_first=False, dtype=int)

# Update X to include the new one-hot encoded city, accommodation type columns, and the encoded Price_bucket
X = df[['star_rating', 'Date_Diff', 'Price_bucket_encoded'] + [col for col in df.columns if 'City_' in col or 'accommadation_type_name_' in col]]
y = df['ADR_USD']

# Ensure y (ADR_USD) is numeric, coercing errors to NaN and dropping any rows where it became NaN
y = pd.to_numeric(y, errors='coerce').dropna()

# After potentially dropping rows in y, ensure X and y still have matching indices and lengths
X = X.loc[y.index]

# Convert any boolean columns in X to integer type for statsmodels compatibility
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

# No constant is added to avoid multicollinearity when all city and accommodation type dummies are included

# Refit the regression model with the updated X
model = sm.OLS(y, X).fit()

print('\n=== Model Summary with City, Accommodation Type, and Price Bucket Encoding ===')
print(model.summary())

print('\n=== Coefficients with City, Accommodation Type, and Price Bucket Encoding ===')
coefficients = model.params
print(coefficients)


=== Model Summary with City, Accommodation Type, and Price Bucket Encoding ===
                            OLS Regression Results                            
Dep. Variable:                ADR_USD   R-squared:                       0.804
Model:                            OLS   Adj. R-squared:                  0.804
Method:                 Least Squares   F-statistic:                     5913.
Date:                Fri, 15 May 2026   Prob (F-statistic):               0.00
Time:                        15:20:13   Log-Likelihood:            -1.7578e+05
No. Observations:               31750   AIC:                         3.516e+05
Df Residuals:                   31727   BIC:                         3.518e+05
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                                                            coef    std err          t      P>|t|      [0.025      0.975]
--------

In [105]:
def predict_adr_usd(star_rating, date_diff, city, accommodation_type_name, price_bucket):
    # Ensure mapping is available (defined in 9eccec92)
    price_bucket_mapping = {'Luxury': 2, 'Budget friendly': 1, 'Low': 0}

    # Initialize a dictionary for the new input
    input_data = {
        'star_rating': [star_rating],
        'Date_Diff': [date_diff],
        'Price_bucket_encoded': [price_bucket_mapping.get(price_bucket, None)] # Use .get to handle unseen values gracefully
    }

    # Create a DataFrame for the new input
    input_df = pd.DataFrame(input_data)

    # Add dummy columns for all possible cities and accommodation types, initialized to 0
    # Use model.params.index to get the list of feature names the model was trained on
    all_feature_names = model.params.index.tolist()

    # Identify city and accommodation type columns from the model's trained features
    all_city_cols = [col for col in all_feature_names if 'City_' in col]
    all_accom_cols = [col for col in all_feature_names if 'accommadation_type_name_' in col]

    for col in all_city_cols:
        input_df[col] = 0
    for col in all_accom_cols:
        input_df[col] = 0

    # Set the appropriate dummy column to 1 based on the input city and accommodation type
    city_col_name = f'City_{city}'
    if city_col_name in input_df.columns:
        input_df[city_col_name] = 1
    else:
        print(f"Warning: City '{city}' not recognized. Prediction might be inaccurate.")
        return None

    accom_col_name = f'accommadation_type_name_{accommodation_type_name}'
    if accom_col_name in input_df.columns:
        input_df[accom_col_name] = 1
    else:
        print(f"Warning: Accommodation type '{accommodation_type_name}' not recognized. Prediction might be inaccurate.")
        return None

    # Ensure Price_bucket_encoded is not None
    if input_df['Price_bucket_encoded'].isnull().any():
        print(f"Warning: Price bucket '{price_bucket}' not recognized. Prediction might be inaccurate.")
        return None

    try:
        # Select and reorder columns to match the training data's X using model.params.index
        X_predict = input_df[model.params.index] # Corrected line: using model.params.index

        # Predict ADR_USD
        predicted_adr = model.predict(X_predict)[0]
        return predicted_adr
    except KeyError as e:
        print(f"Error aligning features for prediction: {e}. Check if all required features are present and correctly named.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during prediction: {e}")
        return None

## Interactive Prediction Input

Use the form below to enter your desired values and get a real-time price prediction.

In [109]:
# @title Enter Values for Prediction
star_rating_input = 4 # @param {type:"number"}
date_diff_input = 10 # @param {type:"integer"}
city_input = "City_B" # @param ['City_A', 'City_B', 'City_C', 'City_D', 'City_E'] {type:"string"}
accommodation_type_input = "Hotel" # @param ['Hotel', 'Resort', 'Apartment', 'Guest house'] {type:"string"}
price_bucket_input = "Luxury" # @param ['Luxury', 'Budget friendly', 'Low'] {type:"string"}

print("--- Interactive Prediction ---")
predicted_adr_interactive = predict_adr_usd(
    star_rating=star_rating_input,
    date_diff=date_diff_input,
    city=city_input,
    accommodation_type_name=accommodation_type_input,
    price_bucket=price_bucket_input
)

if predicted_adr_interactive is not None:
    print(f"Predicted ADR_USD for {city_input}, {star_rating_input} stars, {date_diff_input} days, {accommodation_type_input}, {price_bucket_input} price bucket: ${predicted_adr_interactive:.2f}")
else:
    print("Could not make a prediction with the provided inputs. Please check the city, accommodation type name, and price bucket.")

--- Interactive Prediction ---
Predicted ADR_USD for City_B, 4 stars, 10 days, Hotel, Luxury price bucket: $321.24
